In [1]:
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]

POSITIVE_CONTROLS = [
    ("760.5856", "761.5864"), ("788.6156", "789.6178"), 
    ("810.5997", "811.6054"), ("798.5392", "799.5461"),
    ("813.6858", "814.6872"), ("524.3706", "525.3731"),
    ("522.3557", "523.3569"), ("734.5686", "735.572"),
    ("947.6513", "948.6564"), ("560.3341", "561.3381"),
    ("971.6514", "972.6575"), ("500.3482", "501.3524"), 
    ("484.2952", "485.3011"), ("627.534", "628.5358"), 
    ("455.2904", "456.2954"), ("496.339", "497.3426"),
    ("666.4319", "667.4356"), ("678.4684", "679.4734"), 
    ("703.574", "704.5774"), ("706.5388", "707.5393")
]

NEGATIVE_CONTROLS = [
    ("760.5856", "810.5997"), ("788.6156", "798.5951"), 
    ("810.5997", "792.632"), ("792.632", "524.3706"),
    ("813.6858", "522.3557"), ("524.3706", "947.6513"),
    ("740.4725", "739.5955"), ("739.4661", "740.6021"),
    ("629.5422", "630.6165"), ("579.5304", "592.3947"), 
    ("759.6355", "760.5856"), ("772.6211", "773.5294"),
    ("788.6156", "797.5895"), ("840.641", "842.6652"), 
    ("848.5591", "848.6185"), ("848.5591", "848.6907"),
    ("868.6629", "870.6936"), ("971.6514", "971.6854") 
]

FOLD_CHANGE = 10

# =============================================================================
# 2. SHARED SPATIAL FUNCTIONS (identical to isotope matcher)
# =============================================================================

@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None


def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_morans_i_vectorized(coords: np.ndarray, values: np.ndarray, indices: np.ndarray) -> float:
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


# =============================================================================
# 3. METRIC FUNCTIONS (identical to isotope matcher, all clamped to [0, 1])
# =============================================================================

def compute_coordinate_based_similarity(sig1: SpatialSignature, sig2: SpatialSignature, grid_res: int = 50) -> dict:
    """Compute coordinate-based similarity between two signatures in the same coordinate space.
    All returned values are in [0, 1]."""
    # Direct value correlation — clamp to [0, 1]
    mask = (sig1.raw_values > 0) | (sig2.raw_values > 0)
    value_corr = 0.0
    if mask.sum() > 10:
        r, _ = pearsonr(sig1.raw_values, sig2.raw_values)
        value_corr = max(0.0, r) if not np.isnan(r) else 0.0

    # Importance correlation — clamp to [0, 1]
    imp_corr = 0.0
    if len(sig1.node_importance) > 10:
        r, _ = pearsonr(sig1.node_importance, sig2.node_importance)
        imp_corr = max(0.0, r) if not np.isnan(r) else 0.0

    # Importance IoU — naturally [0, 1]
    imp1 = sig1.node_importance / (sig1.node_importance.max() + 1e-8)
    imp2 = sig2.node_importance / (sig2.node_importance.max() + 1e-8)
    importance_iou = np.minimum(imp1, imp2).sum() / (np.maximum(imp1, imp2).sum() + 1e-8)

    # Intensity ratio consistency — naturally [0, 1]
    intensity_ratio_consistency = compute_intensity_ratio_consistency(sig1.raw_values, sig2.raw_values)

    # Peak colocalization — naturally [0, 1]
    peak_colocalization = compute_peak_colocalization(sig1.raw_values, sig2.raw_values)

    return {
        'value_correlation': value_corr,
        'importance_iou': importance_iou,
        'importance_correlation': imp_corr,
        'intensity_ratio_consistency': intensity_ratio_consistency,
        'peak_colocalization': peak_colocalization
    }


def compute_intensity_ratio_consistency(values1: np.ndarray, values2: np.ndarray, min_intensity_pct: float = 10) -> float:
    """Returns [0, 1]. Higher = more consistent intensity ratio across pixels."""
    thresh1 = np.percentile(values1, min_intensity_pct)
    thresh2 = np.percentile(values2, min_intensity_pct)
    mask = (values1 > thresh1) & (values2 > thresh2)
    if mask.sum() < 10:
        return 0.0
    v1 = values1[mask]
    v2 = values2[mask]
    ratios = np.minimum(v1, v2) / (np.maximum(v1, v2) + 1e-8)
    ratio_mean = ratios.mean()
    ratio_std = ratios.std()
    if ratio_mean > 0:
        cv = ratio_std / ratio_mean
        consistency = 1 / (1 + cv)
    else:
        consistency = 0.0
    return consistency


def compute_peak_colocalization(values1: np.ndarray, values2: np.ndarray, top_pct: float = 20) -> float:
    """Returns [0, 1]. IoU of top-percentile intensity pixels."""
    thresh1 = np.percentile(values1, 100 - top_pct)
    thresh2 = np.percentile(values2, 100 - top_pct)
    peaks1 = values1 >= thresh1
    peaks2 = values2 >= thresh2
    intersection = (peaks1 & peaks2).sum()
    union = (peaks1 | peaks2).sum()
    if union > 0:
        return intersection / union
    return 0.0


def compute_descriptor_similarity(sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
    """All returned values are in [0, 1]."""
    def safe_pearsonr_clamped(a, b):
        """Pearson r clamped to [0, 1]: negative correlation treated as no similarity."""
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return max(0.0, r) if not np.isnan(r) else 0.0
        return 0.0

    spatial_hist_corr = safe_pearsonr_clamped(sig1.spatial_histogram.flatten(), sig2.spatial_histogram.flatten())
    radial_corr = safe_pearsonr_clamped(sig1.radial_profile, sig2.radial_profile)

    # Moran's I in [-1, 1], so abs(a - b) in [0, 2] — clamp result to [0, 1]
    morans_sim = max(0.0, 1 - abs(sig1.morans_i - sig2.morans_i))

    return {
        'spatial_hist_corr': spatial_hist_corr,
        'radial_corr': radial_corr,
        'morans_similarity': morans_sim
    }


def get_8_metrics(sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
    """Extract the 8 metrics used by the isotope matcher's compute_combined_score.
    All metrics guaranteed to be in [0, 1]."""
    coord_sim = compute_coordinate_based_similarity(sig1, sig2)
    desc_sim = compute_descriptor_similarity(sig1, sig2)

    # Value IoU — naturally [0, 1]
    val_iou = np.minimum(sig1.raw_values, sig2.raw_values).sum() / (
        np.maximum(sig1.raw_values, sig2.raw_values).sum() + 1e-8)

    return {
        'intensity_ratio_consistency': coord_sim['intensity_ratio_consistency'],
        'value_correlation': coord_sim['value_correlation'],
        'peak_colocalization': coord_sim['peak_colocalization'],
        'importance_iou': coord_sim['importance_iou'],
        'importance_correlation': coord_sim['importance_correlation'],
        'spatial_hist_corr': desc_sim['spatial_hist_corr'],
        'value_iou': val_iou,
        'morans_similarity': desc_sim['morans_similarity']
    }


# =============================================================================
# 4. SIGNATURE EXTRACTION (identical to isotope matcher)
# =============================================================================

def compute_bio_importance(coords: np.ndarray, values: np.ndarray, nn_indices: np.ndarray) -> np.ndarray:
    neighbor_vals = values[nn_indices[:, 1:]]
    local_var = np.var(neighbor_vals, axis=1)
    lv_min, lv_max = local_var.min(), local_var.max()
    if lv_max > lv_min:
        local_var = (local_var - lv_min) / (lv_max - lv_min)
    else:
        local_var = np.zeros_like(local_var)
    v_min, v_max = values.min(), values.max()
    if v_max > v_min:
        val_norm = (values - v_min) / (v_max - v_min)
    else:
        val_norm = np.zeros_like(values)
    return 0.5 * local_var + 0.5 * val_norm


def extract_signature(coords: np.ndarray, values: np.ndarray, sample_id: str,
                      feature_name: str, nn_indices: np.ndarray) -> SpatialSignature:
    bio_imp = compute_bio_importance(coords, values, nn_indices)
    return SpatialSignature(
        sample_id=sample_id, feature_name=feature_name, feature_type='mz',
        node_importance=bio_imp,
        spatial_histogram=compute_spatial_histogram(coords, values),
        radial_profile=compute_radial_profile(coords, values),
        morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
        coordinates=coords, raw_values=values)


# =============================================================================
# 5. CROSS-VALIDATED ML CALIBRATOR
# =============================================================================

class CrossValidatedCalibrator:
    def run(self):
        print("--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---")
        data_rows = []
        for file_name in tqdm(MSI_SAMPLE_FILES):
            path = os.path.join(MSI_INPUT_FOLDER, file_name)
            adata = sc.read_h5ad(path)
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            msi_data = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X

            # Normalize coordinates for NN (same as isotope matcher)
            coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
            norm_coords = (coords - coord_min) / (coord_max - coord_min + 1e-8)
            k_actual = min(9, len(coords))
            nn_idx = NearestNeighbors(n_neighbors=k_actual).fit(norm_coords).kneighbors(norm_coords)[1]

            var_names = list(adata.var_names)

            for m1, m2 in POSITIVE_CONTROLS + NEGATIVE_CONTROLS:
                if m1 in var_names and m2 in var_names:
                    i1 = var_names.index(m1)
                    i2 = var_names.index(m2)
                    v1 = msi_data[:, i1].flatten() if msi_data.ndim == 2 else msi_data[:, i1]
                    v2 = msi_data[:, i2].flatten() if msi_data.ndim == 2 else msi_data[:, i2]

                    sig1 = extract_signature(coords, v1, file_name, m1, nn_idx)
                    sig2 = extract_signature(coords, v2, file_name, m2, nn_idx)

                    metrics = get_8_metrics(sig1, sig2)
                    metrics['label'] = 1 if (m1, m2) in POSITIVE_CONTROLS else 0
                    metrics['sample_id'] = file_name
                    data_rows.append(metrics)

        df = pd.DataFrame(data_rows)
        samples = df['sample_id'].unique()

        ordered_features = [
            'intensity_ratio_consistency', 'value_correlation', 'peak_colocalization',
            'importance_iou', 'importance_correlation', 'spatial_hist_corr',
            'value_iou', 'morans_similarity'
        ]

        X_full = df[ordered_features]
        y_full = df['label']

        print(f"\n--- PHASE 2: {FOLD_CHANGE}-FOLD CROSS-VALIDATION (BY ANIMAL) ---")
        kf = KFold(n_splits=FOLD_CHANGE, shuffle=True, random_state=42)
        fold_aucs = []
        fold_weights = []

        for train_idx, test_idx in kf.split(samples):
            train_samples = samples[train_idx]
            test_samples = samples[test_idx]

            X_train = df[df['sample_id'].isin(train_samples)][ordered_features]
            y_train = df[df['sample_id'].isin(train_samples)]['label']
            X_test = df[df['sample_id'].isin(test_samples)][ordered_features]
            y_test = df[df['sample_id'].isin(test_samples)]['label']

            model = LogisticRegression(fit_intercept=False, penalty='l2')
            model.fit(X_train, y_train)

            coeffs = np.abs(model.coef_[0])
            normalized_w = coeffs / np.sum(coeffs)
            fold_weights.append(normalized_w)

            y_probs = model.predict_proba(X_test)[:, 1]
            fold_aucs.append(roc_auc_score(y_test, y_probs))

        avg_weights = np.mean(fold_weights, axis=0)
        std_weights = np.std(fold_weights, axis=0)
        avg_auc = np.mean(fold_aucs)

        self.print_thesis_report(ordered_features, avg_weights, std_weights, avg_auc)

    def print_thesis_report(self, names, avg_w, std_w, auc):
        se_w = std_w / np.sqrt(FOLD_CHANGE)

        print("\n" + "=" * 95)
        print("PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS")
        print("=" * 95)
        print(f"{FOLD_CHANGE}-Fold Cross-Validated AUC: {auc:.4f}")
        print("\nValidated Weights (Fractional Mean [0-1] +/- SD and SE):")
        print("-" * 95)
        print(f"{'Metric Name':<35} | {'Weight (0-1)':<14} | {'SD':<10} | {'Std. Error (SE)':<15}")
        print("-" * 95)
        for i in range(len(names)):
            print(f"{names[i]:<35} | {avg_w[i]:>12.4f}   | {std_w[i]:>8.4f}   | {se_w[i]:>12.4f}")
        print("-" * 95)
        print(f"{'TOTAL (Sum)':<35} | {np.sum(avg_w):>12.4f}   |")
        print("=" * 95)
        print(f"Justification: Standard Error (SE) represents the precision of the weight estimate across")
        print(f"all {FOLD_CHANGE} folds. Low SE indicates high confidence in the global weight distribution.")
        print("=" * 95)


if __name__ == "__main__":
    calibrator = CrossValidatedCalibrator()
    calibrator.run()

--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---


100%|██████████| 16/16 [00:03<00:00,  4.02it/s]


--- PHASE 2: 10-FOLD CROSS-VALIDATION (BY ANIMAL) ---

PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS
10-Fold Cross-Validated AUC: 0.9850

Validated Weights (Fractional Mean [0-1] +/- SD and SE):
-----------------------------------------------------------------------------------------------
Metric Name                         | Weight (0-1)   | SD         | Std. Error (SE)
-----------------------------------------------------------------------------------------------
intensity_ratio_consistency         |       0.2231   |   0.0032   |       0.0010
value_correlation                   |       0.0414   |   0.0049   |       0.0016
peak_colocalization                 |       0.0850   |   0.0069   |       0.0022
importance_iou                      |       0.1226   |   0.0038   |       0.0012
importance_correlation              |       0.1948   |   0.0030   |       0.0009
spatial_hist_corr                   |       0.1111   |   0.0056   |       0.0018
value_iou                         